# Phase 3 — Indicator-Variable Applications

Empirical companion to `notes/phase3-indicators.md`. Each of the four classical problems gets:

- the indicator setup recalled in one cell,
- a simulation that converges to the closed-form expectation,
- a plot or table showing the convergence and (where relevant) the distribution shape.

All simulators live in `src/indicators.py`. We treat this notebook as a check on those implementations and a visual demo of the trick.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.indicators import (
    expected_coupon_collector, simulate_coupon_collector,
    expected_hat_check,       simulate_hat_check,
    expected_inversions,      simulate_inversions,
    expected_triangles_gnp,   simulate_triangles_gnp,
)

RNG = np.random.default_rng(seed=20260526)

## 1. Coupon collector — $E[T] = n\,H_n$

Sweep $n \in \{5, 10, 20, 50, 100\}$, run 5{,}000 simulations per value, compare with $n H_n$.

In [ ]:
REPS = 5_000
n_grid = [5, 10, 20, 50, 100]
rows = []
for n in n_grid:
    samples = np.array([simulate_coupon_collector(n, RNG) for _ in range(REPS)], dtype=float)
    mean = samples.mean()
    se   = samples.std(ddof=1) / np.sqrt(REPS)
    rows.append({
        "n":              n,
        "E[T] = n*H_n":   expected_coupon_collector(n),
        "empirical mean": mean,
        "empirical SE":   se,
    })
pd.DataFrame(rows).round(3)

The empirical means sit within a fraction of a standard error of $n H_n$. As $n$ doubles from 50 to 100, $E[T]$ grows by a factor of $\approx 2.31$ — that is the $H_n / H_{n/2}$ ratio, slightly more than linear because the harmonic numbers grow like $\ln n$.

## 2. Hat-check — $E[F] = 1$ regardless of $n$

The striking part. We also verify $\text{Var}(F) = 1$ and show that the distribution converges to $\text{Poisson}(1)$.

In [ ]:
REPS = 20_000
n_grid = [5, 10, 20, 50, 100, 500]
rows = []
for n in n_grid:
    samples = np.array([simulate_hat_check(n, RNG) for _ in range(REPS)], dtype=float)
    rows.append({
        "n":                n,
        "E[F] analytical":  expected_hat_check(n),
        "empirical mean":   samples.mean(),
        "empirical Var(F)": samples.var(ddof=1),
    })
pd.DataFrame(rows).round(4)

Both columns sit at $\approx 1.000$ across the entire range of $n$. Now visualize the distribution of $F$ at $n = 50$ next to the $\text{Poisson}(1)$ PMF.

In [ ]:
from math import factorial

n = 50
REPS = 50_000
samples = np.array([simulate_hat_check(n, RNG) for _ in range(REPS)], dtype=int)

k_vals = np.arange(0, 7)
empirical_pmf = np.array([(samples == k).mean() for k in k_vals])
poisson_pmf   = np.array([np.exp(-1) / factorial(k) for k in k_vals])

fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(len(k_vals))
ax.bar(x - 0.18, empirical_pmf, width=0.36, label=f"empirical (n={n})", alpha=0.85)
ax.bar(x + 0.18, poisson_pmf,   width=0.36, label="Poisson(1)",        alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(k_vals)
ax.set_xlabel("k = number of fixed points")
ax.set_ylabel("P(F = k)")
ax.set_title("Hat-check distribution vs Poisson(1)")
ax.legend()
plt.tight_layout()
plt.show()

The empirical PMF and $\text{Poisson}(1)$ are visually indistinguishable already at $n = 50$ — and the mean and variance are both exactly $1$ at every $n$. The Poisson limit is one of the cleaner classical results in combinatorial probability, and linearity carries the mean part of it.

## 3. Inversions — $E[I] = n(n-1)/4$

In [ ]:
REPS = 2_000
n_grid = [5, 10, 20, 50, 100]
rows = []
for n in n_grid:
    samples = np.array([simulate_inversions(n, RNG) for _ in range(REPS)], dtype=float)
    rows.append({
        "n":                  n,
        "E[I] = n(n-1)/4":    expected_inversions(n),
        "empirical mean":     samples.mean(),
        "empirical SE":       samples.std(ddof=1) / np.sqrt(REPS),
        "max possible C(n,2)": n * (n - 1) // 2,
    })
pd.DataFrame(rows).round(2)

$E[I]$ is exactly half of $\binom{n}{2}$ — a uniform random permutation is, on average, half-sorted. This is the right cost model for insertion sort.

## 4. Triangles in $G(n, p)$ — $E[T] = \binom{n}{3} p^3$

Sweep $n = 30$ across a grid of $p$, and contrast the threshold regime ($p \sim 1/n$) with the dense regime.

In [ ]:
REPS = 1_500
n = 30
p_grid = [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
rows = []
for p in p_grid:
    samples = np.array([simulate_triangles_gnp(n, p, RNG) for _ in range(REPS)], dtype=float)
    rows.append({
        "p":                   p,
        "E[T] = C(n,3) p^3":   expected_triangles_gnp(n, p),
        "empirical mean":      samples.mean(),
        "empirical SE":        samples.std(ddof=1) / np.sqrt(REPS),
        "P(T = 0) empirical":  (samples == 0).mean(),
    })
pd.DataFrame(rows).round(3)

Note the column `P(T = 0)`: at $p = 0.02$, the expected number of triangles is below $0.1$ and roughly $90\%$ of realizations have *zero* triangles. At $p = 0.1$, $E[T]$ is well above $1$ and almost every realization has at least one triangle. The threshold between "typically zero" and "typically present" sits around $p \sim 1/n$ — that is the classical Erdős–Rényi threshold.

## Summary table — all four problems

Linearity gives, in one line each:

In [ ]:
summary = pd.DataFrame([
    {"problem": "Coupon collector", "E[N]": "n * H_n",            "n indicators": "n",         "E[indicator]": "n/(n-i+1)"},
    {"problem": "Hat-check",        "E[N]": "1",                   "n indicators": "n",         "E[indicator]": "1/n"},
    {"problem": "Inversions",       "E[N]": "n(n-1)/4",            "n indicators": "C(n,2)",    "E[indicator]": "1/2"},
    {"problem": "Triangles G(n,p)", "E[N]": "C(n,3) p^3",          "n indicators": "C(n,3)",    "E[indicator]": "p^3"},
])
summary

Same template — define the indicators, find one probability, sum. The dependence structure of the indicators (independent for triangles, *not* independent for hat-check and inversions) is invisible to the mean computation. That is what linearity of expectation, framed this way, does for a working analyst.

Phase 4 takes the same idea and applies it to a 50-person team budget, where the indicator decomposition is replaced by the salary decomposition $\text{Total} = \sum S_i$ and the same logic gives mean stability under arbitrary correlation.